In [1]:
import pandas as pd
import numpy as np

import sys
# ВАЖНО! все задействованные функции должны быть бещ инита в своих папках
MODULE_FULL_PATH = 'C:\ROSTICS-LAB\project'

sys.path.insert(0, MODULE_FULL_PATH)

from paths import *
from simulation_config import common_config, configurations
config = {**common_config, **configurations[0]}
from aa_test import AATest
from tools.uploader import Uploader

<>:6: SyntaxWarning: invalid escape sequence '\R'
<>:6: SyntaxWarning: invalid escape sequence '\R'
C:\Users\gpe9038\AppData\Local\Temp\ipykernel_11616\4044231363.py:6: SyntaxWarning: invalid escape sequence '\R'
  MODULE_FULL_PATH = 'C:\ROSTICS-LAB\project'
C:\Users\gpe9038\AppData\Local\Temp\ipykernel_11616\4044231363.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
kiosk_data = Uploader().upload_from_file(config['kiosk_path'], 'csv') 
kassa_data = pd.DataFrame() 
cc_data = pd.DataFrame()

ab_test = AATest(config, kiosk_data, kassa_data, cc_data)
results = ab_test.execute()

0.0


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 72.55it/s]


0.1


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 73.92it/s]


0.2


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 72.79it/s]


0.30000000000000004


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 73.20it/s]


In [3]:
results

,method,test,mde_aa,first_type_errors_aa,second_type_errors_ab
0,bootstrap,ttest,0.0,0.061,0.836
1,bootstrap,ttest,0.1,0.051,0.165
2,bootstrap,ttest,0.2,0.057,0.028
3,bootstrap,ttest,0.3,0.173,0.000


In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from tqdm import tqdm
from scipy.stats import mannwhitneyu, wilcoxon
import seaborn as sns
import statsmodels.api as sm


from datetime import datetime, timedelta



import sys
# ВАЖНО! все задействованные функции должны быть бещ инита в своих папках
MODULE_FULL_PATH = 'C:\ROSTICS-LAB\project'

sys.path.insert(0, MODULE_FULL_PATH)

from paths import *
from simulation_config import common_config, configurations
config = {**common_config, **configurations[0]}
from data_preparation import PrepareData
from splitter.splitter import Splitter
from statistics_tools.simulations import Simulations
from preprocessing.methods import Methods

df = pd.read_csv(config['kiosk_path'])
aggregate = PrepareData.get(df, 'kiosk', config['name'], (datetime.strptime(config['start_date'], '%Y-%m-%d') - timedelta(days=config['test_duration']+1)).strftime('%Y-%m-%d'), (datetime.strptime(config['start_date'], '%Y-%m-%d') + timedelta(days=config['test_duration']+1)).strftime('%Y-%m-%d'))

splitter = Splitter(config)
simulations = Simulations(config)

results_list = []

for mde in np.arange(config['MDE_start'], config['MDE_finish'] + config['MDE_step'], config['MDE_step']):
    for i in range(1):
        aggregate, aggregate_test, aggregate_control = splitter.get_split(aggregate) 
        
        aggregate_dict = {
            'aggregate': aggregate,
            'aggregate_test_before': aggregate_test[aggregate_test['status'] == 'before'],
            'aggregate_control_before': aggregate_control[aggregate_control['status'] == 'before'],
            'aggregate_test_after': aggregate_test[aggregate_test['status'] == 'after'],
            'aggregate_control_after': aggregate_control[aggregate_control['status'] == 'after']
        }
        
        aggregate_test_after, aggregate_control_after = simulations.generate_effect(aggregate_dict, mde)
        
        test_method = Methods(config)
        
        aggregate_dict_copy = {
            'aggregate': aggregate,
            'aggregate_test_before': aggregate_dict['aggregate_test_before'],
            'aggregate_control_before': aggregate_dict['aggregate_control_before'],
            'aggregate_test_after': aggregate_test_after,
            'aggregate_control_after': aggregate_control_after
        }
        print(mde)
        results = test_method.run(aggregate_dict_copy) 
        results_list.append({'mde': mde, 'results': results})

# Создание DataFrame из списка результатов
results_df = pd.DataFrame(results_list)

# Преобразование словаря results в отдельные столбцы для каждого элемента в словаре
expanded_results = results_df['results'].apply(pd.Series)

# Добавление столбца mde к расширенному датафрейму
expanded_results['mde'] = results_df['mde']
expanded_results = expanded_results[['mde', 'results_aa', 'results_ab']]
expanded_results_aa = expanded_results['results_aa'].apply(pd.Series)
expanded_results_ab = expanded_results['results_ab'].apply(pd.Series)
expanded_results_aa['mde'] = results_df['mde']
expanded_results_ab['mde'] = results_df['mde']

# Объединение DataFrame expanded_results_aa и expanded_results_ab по индексу
expanded_results_merged = expanded_results_aa.join(
    expanded_results_ab, 
    lsuffix='_aa', 
    rsuffix='_ab'
).reset_index(drop=True)

averaged_results_by_mde = expanded_results_merged.groupby('mde_aa').mean().reset_index()
averaged_results_by_mde['method'] = config['method']
averaged_results_by_mde['test'] = config['test']
averaged_results_by_mde[['method', 'test', 'mde_aa','first_type_errors_aa','second_type_errors_ab']]



<>:16: SyntaxWarning: invalid escape sequence '\R'
<>:16: SyntaxWarning: invalid escape sequence '\R'
C:\Users\gpe9038\AppData\Local\Temp\ipykernel_20584\4025770044.py:16: SyntaxWarning: invalid escape sequence '\R'
  MODULE_FULL_PATH = 'C:\ROSTICS-LAB\project'
C:\Users\gpe9038\AppData\Local\Temp\ipykernel_20584\4025770044.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


0.0


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 73.14it/s]


0.1


AБ Тест: 100%|██████████| 1000/1000 [00:13<00:00, 73.34it/s]


,method,test,mde_aa,first_type_errors_aa,second_type_errors_ab
0,bootstrap,ttest,0.0,0.088,0.747
1,bootstrap,ttest,0.1,0.057,0.173


In [1]:
aggregate_dict

NameError: name 'aggregate_dict' is not defined

In [14]:
aggregate_dict = {
    'aggregate': aggregate,
    'aggregate_test_before': aggregate_test[aggregate_test['status'] == 'before'],
    'aggregate_control_before': aggregate_control[aggregate_control['status'] == 'before'],
    'aggregate_test_after': aggregate_test[aggregate_test['status'] == 'after'],
    'aggregate_control_after': aggregate_control[aggregate_control['status'] == 'after']
}

sample_size_dict = {
    'aggregate_size': aggregate[config['aggregator']].count(),
    'aggregate_test_before_size': aggregate_dict['aggregate_test_before'][config['aggregator']].count(),
    'aggregate_control_before_size': aggregate_dict['aggregate_control_before'][aggregate_control['status'] == 'before'][config['aggregator']].count(),
    'aggregate_test_after_size': aggregate_dict['aggregate_test_after'][aggregate_test['status'] == 'after'][config['aggregator']].count(),
    'aggregate_control_after_size': aggregate_dict['aggregate_control_after'][aggregate_control['status'] == 'after'][config['aggregator']].count()
}

{'aggregate_size': 88640,
 'aggregate_test_before_size': 19840,
 'aggregate_control_before_size': 19840,
 'aggregate_test_after_size': 19840,
 'aggregate_control_after_size': 19776}

In [5]:
# Параметры для симуляции
mu_control = 2500
std = 80
real_effect = 2.536

def generate_simulated_values(data, group, condition_date, mu, std):
    """
    Генерирует симулированные значения для заданной группы и условия даты.
    
    Параметры:
    - data (DataFrame): Данные, для которых необходимо сгенерировать значения.
    - group (array): Массив идентификаторов группы, для которой генерируются значения.
    - condition_date (Series): Условие, определяющее даты, для которых необходимо сгенерировать значения.
    - mu (float): Среднее значение распределения, из которого генерируются значения.
    - std (float): Стандартное отклонение распределения.
    
    Возвращает:
    - values (ndarray): Сгенерированные значения.
    """
    size = data[(data['restraunt_id'].isin(group)) & condition_date][config['aggregator']].count()
    values = np.random.normal(mu, std, size)
    
    data.loc[(data['restraunt_id'].isin(group)) & condition_date, config['aggregator']] = values

def simulate_values(data, period, control_group, test_group, mu_control, std, real_effect):
    """
    Симулирует значения для тестовой и контрольной групп до и после даты начала эксперимента.
    
    Параметры:
    - data (DataFrame): Данные для симуляции.
    - period (str): Период для симуляции ('pre' для до, 'post' для после даты начала).
    - control_group (array): Массив идентификаторов контрольной группы.
    - test_group (array): Массив идентификаторов тестовой группы.
    - mu_control (float): Среднее значение для контрольной группы.
    - std (float): Стандартное отклонение для генерации значений.
    - real_effect (float): Реальный эффект, который необходимо добавить к тестовой группе после даты начала.
    """

    if period == 'pre':
        condition_date = data.event_date < common_config['start_date']
        effect = 0
    else:
        condition_date = data.event_date > common_config['start_date']
        effect = real_effect

    for group, mu in zip([test_group, control_group], [mu_control + effect, mu_control]):
        generate_simulated_values(data, group, condition_date, mu, std)
        
def simulate_effect(data, period, control_group, test_group, mu_control, std, real_effect):
    """
    Симулирует значения для тестовой и контрольной групп до и после даты начала эксперимента.
    
    Параметры:
    - data (DataFrame): Данные для симуляции.
    - period (str): Период для симуляции ('pre' для до, 'post' для после даты начала).
    - control_group (array): Массив идентификаторов контрольной группы.
    - test_group (array): Массив идентификаторов тестовой группы.
    - mu_control (float): Среднее значение для контрольной группы.
    - std (float): Стандартное отклонение для генерации значений.
    - real_effect (float): Реальный эффект, который необходимо добавить к тестовой группе после даты начала.
    """

    if period == 'pre':
        condition_date = data.event_date < common_config['start_date']
        effect = 0
    else:
        condition_date = data.event_date > common_config['start_date']
        effect = real_effect

    for group, mu in zip([test_group, control_group], [mu_control + effect, mu_control]):
        

def calculate_effects(data):
    """
    Рассчитывает реальный и ожидаемый эффекты на основе данных.
    
    Параметры:
    - data (DataFrame): Данные для анализа.
    - control_group (array): Массив идентификаторов контрольной группы.
    - test_group (array): Массив идентификаторов тестовой группы.
    """
    real_effect_percentage = round(real_effect * 100 / data[data.event_date < common_config['start_date']][config['aggregator']].mean(), 2)
    print(f'{real_effect_percentage}% - РЕАЛЬНЫЙ эффект')
    print(f'{real_effect} - РЕАЛЬНЫЙ эффект')

    # Расчет размера выборки по MDE
    calculate_sample_size(data)
def calculate_sample_size(data):
    """
    Рассчитывает размер выборки на основе минимально детектируемого эффекта (MDE) и данных.
    
    Параметры:
    - data (DataFrame): Данные для расчета.
    - control_group (array): Массив идентификаторов контрольной группы.
    - test_group (array): Массив идентификаторов тестовой группы.
    """
    t_alpha = stats.norm.ppf(1 - config['alpha'] / 2)
    t_beta = stats.norm.ppf(config['beta'])
    effect = config['MDE'] / 100 * data[data.event_date < common_config['start_date']][config['aggregator']].mean()
    sample_size = ((t_alpha + t_beta) ** 2 * (std ** 2)) / effect ** 2
    print(f'{config["MDE"]}% - ожидаемый эффект')
    print(f'{effect} - ожидаемый эффект в единицах измерения')
    print(f'{sample_size} - размер выборки для достижения ожидаемого эффекта')

def print_results(data, control_group, test_group):
    """
    Выводит результаты анализа, включая средние значения и дисперсии до и после даты начала, а также информацию о разделении на группы.
    
    Параметры:
    - data (DataFrame): Данные для вывода результатов.
    - control_group (array): Массив идентификаторов контрольной группы.
    - test_group (array): Массив идентификаторов тестовой группы.
    - clusters (DataFrame): Данные о кластерах.
    """
    
    # Вывод средних значений и дисперсий до и после start_date
    print('До start_date:')
    print(f'Среднее значение в контрольной группе: {data[(data.group == "control") & (data.event_date < common_config["start_date"])][config["aggregator"]].mean()}')
    print(f'Среднее значение в тестовой группе: {data[(data.group == "test") & (data.event_date < common_config["start_date"])][config["aggregator"]].mean()}')
    print('После start_date:')
    print(f'Среднее значение в контрольной группе: {data[(data.group == "control") & (data.event_date > common_config["start_date"])][config["aggregator"]].mean()}')
    print(f'Среднее значение в тестовой группе: {data[(data.group == "test") & (data.event_date > common_config["start_date"])][config["aggregator"]].mean()}')

    # Вывод информации о разделении на группы
    print(f'Количество ресторанов в контрольной группе: {len(control_group)}')
    print(f'Количество ресторанов в тестовой группе: {len(test_group)}')

# Присваивание групп до и после start_date
data.loc[(data['restraunt_id'].isin(test_group)) & (data.event_date < common_config['start_date']), 'group'] = 'test'
data.loc[(data['restraunt_id'].isin(control_group)) & (data.event_date < common_config['start_date']), 'group'] = 'control'
data.loc[(data['restraunt_id'].isin(test_group)) & (data.event_date >= common_config['start_date']), 'group'] = 'test'
data.loc[(data['restraunt_id'].isin(control_group)) & (data.event_date >= common_config['start_date']), 'group'] = 'control'

clusters.loc[clusters['restraunt_id'].isin(test_group), 'group'] = 'test'
clusters.loc[clusters['restraunt_id'].isin(control_group), 'group'] = 'control'
clusters = clusters.drop_duplicates().dropna().sort_values(['strat', 'group'])
clusters = clusters.merge(clusters.groupby('strat').size().rename('strat_size'), on='strat')

# Симуляция значений до и после start_date
np.random.seed(44)
simulate_values(data, 'pre', control_group, test_group, mu_control, std, real_effect)
simulate_values(data, 'post', control_group, test_group, mu_control, std, real_effect)

# Расчет реального и ожидаемого эффектов
calculate_effects(data)

# Вывод результатов
print_results(data, control_group, test_group)

IndentationError: expected an indented block after 'for' statement on line 70 (2619676284.py, line 73)

In [ ]:
data

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,event_date,restraunt_id,system_identification_only_showed,open_auth_count,tap_email_count,tap_sms_count,sessions,orders,auth_orders,conversion_rate,auth_conversion_rate,revenue,avg_product_count,strat,group
0,0,0,0,12,2023-10-01,74013270,0,0,0,0,927,2357.580255,7,0.745415,0.007551,329079.0,3.221418,2.0_0.0,control
1,1,1,1,25,2023-10-01,74013406,0,0,0,0,340,2491.473038,1,0.682353,0.002941,98883.0,3.409483,4.0_2.0,control
2,2,2,6,107,2023-10-01,74020492,0,0,0,0,1094,2446.533034,8,0.818099,0.007313,434570.0,3.526257,2.0_0.0,control
3,3,3,10,164,2023-10-01,74020660,0,0,0,0,173,2390.032230,4,0.826590,0.023121,61988.0,3.153846,4.0_0.0,control
4,4,4,12,229,2023-10-01,74020828,0,0,0,0,423,2524.025433,4,0.822695,0.009456,168693.0,3.477011,3.0_3.0,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188603,47147,12319,6769,156174,2024-02-24,74321670,0,0,0,0,575,2357.878165,30,0.853913,0.052174,216844.0,2.879837,7.0_0.0,control
188604,47148,12320,6770,156195,2024-02-24,74321719,0,0,0,0,262,2469.330401,1,0.660305,0.003817,71870.0,2.861272,4.0_0.0,control
188605,47149,12321,6771,156197,2024-02-24,74321722,0,0,0,0,463,2689.146719,7,0.881210,0.015119,224576.0,3.502451,9.0_1.0,control
188606,47150,12322,6772,156260,2024-02-24,74321861,0,0,0,0,215,2430.611430,1,0.758140,0.004651,101485.0,4.006135,0.0_1.0,test


In [ ]:
def ttest(a, b, t_real):
    delta = b.mean() - a.mean()
    std = (a.var() / len(a) + b.var() / len(b)) ** 0.5
    t = delta / std
    pvalue = 2 * (1 - stats.norm.cdf(np.abs(t) - np.abs(t_real)))
    return t, pvalue, delta

def perform_bootstrap_test(data, config):
    first_type_errors = []
    second_type_errors = []
    deltas_aa = []
    deltas_ab = []
    pvalues_aa = []
    pvalues_ab = []
    t_aa_arr = []
    t_ab_arr = []

    # Получение размеров выборок
    sample_size_control_pre = data[(data.group == 'control') & (data.event_date < common_config['start_date'])][config['aggregator']].count()
    sample_size_test_pre = data[(data.group == 'test') & (data.event_date < common_config['start_date'])][config['aggregator']].count()
    sample_size_test_post = data[(data.group == 'test') & (data.event_date >= common_config['start_date'])][config['aggregator']].count()

    for _ in tqdm(range(300)):
        # Использование метода .sample() один раз для каждой группы и периода
        control_aa = data[(data.group == 'control') & (data.event_date < common_config['start_date'])].sample(n=sample_size_control_pre, replace=True)
        control_ab = data[(data.group == 'control') & (data.event_date >= common_config['start_date'])].sample(n=sample_size_control_pre, replace=True)
        test_aa = data[(data.group == 'test') & (data.event_date < common_config['start_date'])].sample(n=sample_size_test_pre, replace=True)
        test_ab = data[(data.group == 'test') & (data.event_date >= common_config['start_date'])].sample(n=sample_size_test_post, replace=True)

        # Вычисление t и pvalue один раз для каждой пары
        t_aa, pvalue_aa, delta_aa = ttest(control_aa[config['aggregator']], test_aa[config['aggregator']], 0)
        t_ab, pvalue_ab, delta_ab = ttest(control_ab[config['aggregator']], test_ab[config['aggregator']], 0)

        # Обновление списков результатов
        first_type_errors.append(pvalue_aa < config['alpha'])
        second_type_errors.append(pvalue_ab >= config['alpha'])
        t_aa_arr.append(t_aa)
        t_ab_arr.append(t_ab)
        deltas_aa.append(delta_aa)
        deltas_ab.append(delta_ab)
        pvalues_aa.append(pvalue_aa)
        pvalues_ab.append(pvalue_ab)

    return first_type_errors, second_type_errors, deltas_aa, deltas_ab, pvalues_aa, pvalues_ab, t_aa_arr, t_ab_arr

def print_statistics(part_first_type_errors, part_second_type_errors, variation, sample_size, test_duration, MDE):
    """
    Функция для печати статистических результатов.
    """
    print('\n')
    print(f'MDE = {MDE}')
    print(f'part_first_type_errors = {part_first_type_errors:0.3f}')
    print(f'part_second_type_errors = {part_second_type_errors:0.3f}')
    print(f'variation = {variation:0.3f}')
    print(f'sample_size = {sample_size:0.3f}')
    print(f'test duration = {test_duration:0.3f}')
    
def calculate_statistics(data, config, first_type_errors, second_type_errors):
    """
    Расчет статистических показателей и подготовка данных для вывода.
    """
    part_first_type_errors = np.mean(first_type_errors)
    part_second_type_errors = np.mean(second_type_errors)
    effect = config['MDE'] / 100 * data[data.event_date < common_config['start_date']][config['aggregator']].mean()
    t_alpha = stats.norm.ppf(1 - config['alpha'] / 2, loc=0, scale=1)
    t_beta = stats.norm.ppf(1 - config['beta'], loc=0, scale=1)
    sample_size = int((t_alpha + t_beta) ** 2 * data[data.event_date < common_config['start_date']][config['aggregator']].var() / (effect ** 2))
    variation = data[data.event_date < common_config['start_date']][config['aggregator']].var()
    test_duration = sample_size / data.restraunt_id.nunique()

    # Вызов функции печати с подготовленными данными
    print_statistics(part_first_type_errors, part_second_type_errors, variation, sample_size, test_duration, config["MDE"])
    
# Выполнение бутстрап-тестов
first_type_errors, second_type_errors, deltas_aa, deltas_ab, pvalues_aa, pvalues_ab, t_aa_arr, t_ab_arr = perform_bootstrap_test(
    data, 
    config
)

# Расчет и вывод статистик
calculate_statistics(data, config, first_type_errors, second_type_errors)



100%|██████████| 300/300 [00:47<00:00,  6.30it/s]



MDE = 6
part_first_type_errors = 0.223
part_second_type_errors = 0.027
variation = 6370.405
sample_size = 2.000
test duration = 0.091


In [ ]:
clusters

,restraunt_id,strat,group,strat_size
0,74321861,0.0_1.0,test,1
1,74321865,0.0_6.0,test,1
2,74215106,1.0_1.0,control,1
3,74020492,2.0_0.0,control,3
4,74013270,2.0_0.0,control,3
5,74021008,2.0_0.0,test,3
6,74020920,3.0_0.0,test,1
7,74020828,3.0_3.0,control,1
8,74020660,4.0_0.0,control,5
9,74020871,4.0_0.0,control,5


In [ ]:
data

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,event_date,restraunt_id,system_identification_only_showed,open_auth_count,tap_email_count,tap_sms_count,sessions,orders,auth_orders,conversion_rate,auth_conversion_rate,revenue,avg_product_count,strat,group
0,0,0,0,12,2023-10-01,74013270,0,0,0,0,927,2357.580255,7,0.745415,0.007551,329079.0,3.221418,2.0_0.0,control
1,1,1,1,25,2023-10-01,74013406,0,0,0,0,340,2491.473038,1,0.682353,0.002941,98883.0,3.409483,4.0_2.0,control
2,2,2,6,107,2023-10-01,74020492,0,0,0,0,1094,2446.533034,8,0.818099,0.007313,434570.0,3.526257,2.0_0.0,control
3,3,3,10,164,2023-10-01,74020660,0,0,0,0,173,2390.032230,4,0.826590,0.023121,61988.0,3.153846,4.0_0.0,control
4,4,4,12,229,2023-10-01,74020828,0,0,0,0,423,2524.025433,4,0.822695,0.009456,168693.0,3.477011,3.0_3.0,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188603,47147,12319,6769,156174,2024-02-24,74321670,0,0,0,0,575,2357.878165,30,0.853913,0.052174,216844.0,2.879837,7.0_0.0,control
188604,47148,12320,6770,156195,2024-02-24,74321719,0,0,0,0,262,2469.330401,1,0.660305,0.003817,71870.0,2.861272,4.0_0.0,control
188605,47149,12321,6771,156197,2024-02-24,74321722,0,0,0,0,463,2689.146719,7,0.881210,0.015119,224576.0,3.502451,9.0_1.0,control
188606,47150,12322,6772,156260,2024-02-24,74321861,0,0,0,0,215,2430.611430,1,0.758140,0.004651,101485.0,4.006135,0.0_1.0,test


In [ ]:
a = data.copy().drop(columns=['strat', 'group'])
a.event_date = a.event_date.astype(str).str.replace('-', '')
a.to_csv(test_data_path)

clusters['strat'] = 0
clusters.to_csv(test_stratification_groups_path)

In [ ]:
sns.histplot(test_aa[config['aggregator']], bins=100)
sns.histplot(control_aa[config['aggregator']], bins=100)

: 

In [ ]:
print(np.abs(control_before.action_order_success.mean() - test_before.action_order_success.mean()))
print(np.abs(np.mean(deltas_aa)))

: 

In [ ]:
ttest(control_before.action_order_success, test_before.action_order_success, 0)[0]

: 

In [ ]:
sns.histplot(t_aa_arr + ttest(control_before.action_order_success, test_before.action_order_success, 0)[0], bins=100)

: 

In [ ]:
(t_aa_arr + ttest(control_before.action_order_success, test_before.action_order_success, 0)[0]).mean()

: 

In [ ]:
np.mean(deltas_ab)

: 

In [ ]:
t_aa_arr + ttest(control_before.action_order_success)

: 

In [ ]:
sns.histplot(pvalues_aa, bins=20)

: 

In [ ]:
sns.displot(
    {'deltas': deltas},
    kind='kde'
)

: 

In [ ]:
data[data[config['aggregator']] > 0][config['aggregator']].hist(bins=50)

: 

In [ ]:
np.log(data[data[config['aggregator']] > 0][config['aggregator']]).hist(bins=50)

: 

In [ ]:
data[config['aggregator']]

: 

In [ ]:
sm.qqplot(np.array(deltas))
sm.qqplot(data[(data[config['aggregator']] > 0) & (data.event_date < common_config['start_date'])][config['aggregator']])
#sm.qqplot(np.log(data[data[config['aggregator']] > 0][config['aggregator']]))

: 

In [ ]:
import scipy.stats as stats

#Levene's test centered at the median
stats.levene(control_after[config['aggregator']].dropna(), test_after[config['aggregator']].dropna(), center='mean')


: 

In [ ]:
t = (control_after.action_order_success.mean() - test_after.action_order_success.mean()) / \
  np.sqrt(control_after.action_order_success.var()/control_after.action_order_success.count() + test_after.action_order_success.var()/test_after.action_order_success.count())
t

: 

In [ ]:
pvalue = 2 * (1 - stats.norm.cdf(np.abs(t)))
pvalue

: 